# Google Places Extraction

Geocodes the pharmacy list against the Google Places Text Search API to retrieve `place_id`, matched name, matched address, and coordinates.

`PHARMACIES_COMBINED.csv` is a pre-existing input produced by a separate PDF scraping process across multiple source lists. This notebook takes it as-is, deduplicates it, and runs the Places API against the result.

Set `PILOT_N` to an integer to run only that many records (useful for validating the pipeline before a full run). Set it to `None` to run all remaining records.

Outputs: `PHARMACIES_FINAL_DEDUPED.csv`, `PHARMACIES_places_full_results.csv`

## Imports and configuration

In [ ]:
import os, time, re
import pandas as pd
import numpy as np
import requests

COMBINED_PATH   = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_COMBINED.csv'
DEDUPED_PATH    = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_FINAL_DEDUPED.csv'
EXISTING_PATH   = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_pilot_results.csv'
OUTPUT_PATH     = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_full_results.csv'
CHECKPOINT_PATH = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_full_results_checkpoint.csv'
HOSPITAL_SHP    = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Data from DAIR\Hospitals\hospitals_kzn_gp.shp'
HOSPITAL_OUT    = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_with_hospital_coords.csv'

API_KEY = os.getenv('GOOGLE_MAPS_API_KEY')
if not API_KEY:
    raise ValueError('GOOGLE_MAPS_API_KEY not found in environment variables.')

PILOT_N       = None    # set to e.g. 100 for a pilot run; None = full run
SLEEP_SECONDS = 0.2
SAVE_EVERY    = 100

print('Config ready.')

## Load and deduplicate pharmacy list

Groups records by `PRACTICE_NUM` and merges entries with identical or near-identical addresses. Records without a practice number are kept as-is.

In [ ]:
df = pd.read_csv(COMBINED_PATH)
print(f'Loaded {len(df):,} rows from PHARMACIES_COMBINED.csv')

# Normalise practice number
df['PRACTICE_NUM'] = (
    df['PRACTICE_NUM'].astype(str).str.strip()
    .replace({'': np.nan, 'nan': np.nan})
)

def normalize_address(x):
    if pd.isna(x):
        return ''
    return str(x).lower().replace(',', '').strip()

df['addr_norm'] = df['ADDRESS'].apply(normalize_address)

# Sanity check for conflicting records before dedup
check = (
    df[df['PRACTICE_NUM'].notna()]
    .groupby('PRACTICE_NUM')
    .agg(n_rows=('PRACTICE_NUM', 'count'),
         n_addresses=('ADDRESS', 'nunique'),
         n_names=('PRACTICE_NAME', 'nunique'))
    .reset_index()
)
conflicts = check[(check['n_addresses'] > 1) | (check['n_names'] > 1)]
print(f'Practice numbers with conflicting addresses/names: {len(conflicts)}')
display(conflicts.head())

In [ ]:
def dedupe_group(group):
    # Single row needs no merging
    if len(group) == 1:
        return group
    unique_addrs = group['addr_norm'].unique()
    if len(unique_addrs) == 1:
        return group.iloc[[0]]
    # Keep group if addresses are too different to merge safely
    base    = unique_addrs[0]
    similar = all(base[:15] in a for a in unique_addrs)
    if similar:
        return group.iloc[[0]]
    return group

with_id    = df[df['PRACTICE_NUM'].notna()]
without_id = df[df['PRACTICE_NUM'].isna()]

final_df = pd.concat([
    with_id.groupby('PRACTICE_NUM', group_keys=False).apply(dedupe_group),
    without_id,
], ignore_index=True)

print(f'Original : {len(df):,}')
print(f'After dedup: {len(final_df):,}')

final_df.to_csv(DEDUPED_PATH, index=False)
print(f'Saved -> {DEDUPED_PATH}')

## Helper functions

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None


def build_query(row):
    # Format: 'Pharmacy Name, Street Address, City, Province, South Africa'
    parts = [p for p in [
        clean_text(row.get('PRACTICE_NAME')),
        clean_text(row.get('ADDRESS')),
        clean_text(row.get('CITY')),
        clean_text(row.get('PROVINCE')),
        'South Africa',
    ] if p]
    return ', '.join(parts)


def search_place_text(query, api_key):
    url    = 'https://maps.googleapis.com/maps/api/place/textsearch/json'
    params = {'query': query, 'type': 'pharmacy', 'key': api_key}
    r      = requests.get(url, params=params, timeout=30)
    try:
        data = r.json()
    except Exception:
        return {
            'http_status': r.status_code, 'api_status': 'BAD_JSON',
            'place_id': None, 'matched_name': None, 'matched_address': None,
            'lat': None, 'lng': None, 'types': None, 'raw_error': r.text[:500],
        }
    status = data.get('status')
    if status != 'OK':
        return {
            'http_status': r.status_code, 'api_status': status,
            'place_id': None, 'matched_name': None, 'matched_address': None,
            'lat': None, 'lng': None, 'types': None, 'raw_error': str(data)[:500],
        }
    result = data['results'][0]
    return {
        'http_status':     r.status_code,
        'api_status':      status,
        'place_id':        result.get('place_id'),
        'matched_name':    result.get('name'),
        'matched_address': result.get('formatted_address'),
        'lat':             result.get('geometry', {}).get('location', {}).get('lat'),
        'lng':             result.get('geometry', {}).get('location', {}).get('lng'),
        'types':           ', '.join(result.get('types', [])) if result.get('types') else None,
        'raw_error':       None,
    }


def needs_review(row):
    if row['api_status'] != 'OK': return True
    if pd.isna(row['place_id']): return True
    if pd.isna(row['lat']) or pd.isna(row['lng']): return True
    return False


print('Helpers defined.')

## Build query strings and identify remaining records

Loads existing results (pilot or previous checkpoint) and skips query strings already completed.

In [ ]:
places_df = final_df.copy()
places_df['query_string'] = places_df.apply(build_query, axis=1)
places_df = places_df[
    places_df['query_string'].notna() &
    (places_df['query_string'].str.len() > 0)
].copy()

print(f'Rows with usable query string: {len(places_df):,}')

# Load previous results and skip already-completed queries
if os.path.exists(EXISTING_PATH):
    existing_results = pd.read_csv(EXISTING_PATH)
    if 'query_string' not in existing_results.columns:
        raise ValueError('Existing results file is missing the query_string column.')
    already_done = set(existing_results['query_string'].dropna())
    remaining_df = places_df[~places_df['query_string'].isin(already_done)].copy()
    print(f'Already completed: {len(already_done):,}')
else:
    existing_results = pd.DataFrame()
    remaining_df     = places_df.copy()

# Apply pilot limit if set
if PILOT_N is not None:
    remaining_df = remaining_df.head(PILOT_N).copy()

print(f'Remaining to query: {len(remaining_df):,}')

## Run API calls

Queries the Places API for each remaining record. Checkpoints every `SAVE_EVERY` rows so the run is resumable.

In [ ]:
results = []

for n, (_, row) in enumerate(remaining_df.iterrows(), start=1):
    query  = row['query_string']
    result = search_place_text(query, API_KEY)

    results.append({
        'PHARMACY_ID':   row.get('PHARMACY_ID'),
        'PRACTICE_NUM':  row.get('PRACTICE_NUM'),
        'PRACTICE_NAME': row.get('PRACTICE_NAME'),
        'ADDRESS':       row.get('ADDRESS'),
        'CITY':          row.get('CITY'),
        'PROVINCE':      row.get('PROVINCE'),
        'PHONE':         row.get('PHONE'),
        'query_string':  query,
        **result,
    })

    if n % 100 == 0:
        print(f'Processed {n:,} / {len(remaining_df):,}')

    if n % SAVE_EVERY == 0:
        chk_new  = pd.DataFrame(results)
        if len(chk_new) > 0:
            chk_new['needs_review'] = chk_new.apply(needs_review, axis=1)
        chk_combined = pd.concat([existing_results, chk_new], ignore_index=True)
        chk_combined = chk_combined.drop_duplicates(subset=['query_string'], keep='first')
        chk_combined.to_csv(CHECKPOINT_PATH, index=False)
        print(f'  Checkpoint saved ({len(chk_combined):,} total)')

    time.sleep(SLEEP_SECONDS)

print(f'Done. {len(results):,} new records queried this run.')

## Combine and save

In [ ]:
new_results = pd.DataFrame(results)
if len(new_results) > 0:
    new_results['needs_review'] = new_results.apply(needs_review, axis=1)

combined = pd.concat([existing_results, new_results], ignore_index=True)
combined = combined.drop_duplicates(subset=['query_string'], keep='first')
combined.to_csv(OUTPUT_PATH, index=False)

print(f'Combined total: {len(combined):,} rows')
print(f'Saved -> {OUTPUT_PATH}')
print()
print('API status counts:')
print(combined['api_status'].value_counts(dropna=False))
print()
print('Needs review counts:')
print(combined['needs_review'].value_counts(dropna=False))

## Hospital coordinate fill-in

For pharmacies still missing coordinates, attempts to fill them from the hospital shapefile by matching on cleaned name. Only fills where the pharmacy name strongly matches a hospital name — this covers hospital-based pharmacies that Places API did not resolve.

In [ ]:
import geopandas as gpd

pharm = pd.read_csv(OUTPUT_PATH)
hosp  = gpd.read_file(HOSPITAL_SHP)

pharm.columns = pharm.columns.str.lower().str.strip()
hosp.columns  = hosp.columns.str.lower().str.strip()

pharm['lat'] = pd.to_numeric(pharm['lat'], errors='coerce')
pharm['lng'] = pd.to_numeric(pharm['lng'], errors='coerce')

def clean_name(x):
    if pd.isna(x):
        return None
    x = str(x).lower().strip().replace('&', ' and ')
    x = re.sub(r'[^a-z0-9\s]', ' ', x)
    x = re.sub(r'\b(hospital|clinic|medical|centre|center|academic|district|regional|provincial)\b', '', x)
    return re.sub(r'\s+', ' ', x).strip()

pharm['name_key'] = pharm['practice_name'].map(clean_name)
hosp['name_key']  = hosp['name'].map(clean_name)

hosp_lookup = (
    hosp[['name', 'name_key', 'latitude', 'longitude']]
    .dropna(subset=['name_key'])
    .drop_duplicates(subset=['name_key'])
    .rename(columns={'name': 'hospital_name_shp', 'latitude': 'lat_shp', 'longitude': 'lng_shp'})
)

merged = pharm.merge(hosp_lookup, on='name_key', how='left')

merged['filled_from_hospital'] = merged['lat'].isna() & merged['lat_shp'].notna()
merged['lat'] = merged['lat'].fillna(merged['lat_shp'])
merged['lng'] = merged['lng'].fillna(merged['lng_shp'])

print(f'Total rows          : {len(merged):,}')
print(f'Hospital name matches: {merged["hospital_name_shp"].notna().sum():,}')
print(f'Coordinates filled  : {merged["filled_from_hospital"].sum():,}')

qa_cols = ['practice_name', 'hospital_name_shp', 'lat', 'lng', 'lat_shp', 'lng_shp', 'filled_from_hospital']
print()
print('Sample matches:')
display(merged.loc[merged['hospital_name_shp'].notna(), qa_cols].head(15))

merged.to_csv(HOSPITAL_OUT, index=False)
print(f'\nSaved -> {HOSPITAL_OUT}')